# 01. Preparación de SQLite y TF-IDF para consultas heterogéneas

Esta libreta prepara los datos locales que usará la siguiente libreta de consultas con LLM.

**Objetivo:** crear únicamente tres archivos de salida:

1. `UNAM_Completo_2024_2025.sqlite`  
   Base SQLite con una sola tabla canónica: `articulos_autores`.

2. `tfidf_articulos.joblib`  
   Índice TF-IDF por artículo, construido con `Titulo`, `Keywords` y `Abstract`.

3. `tfidf_autores.joblib`  
   Índice TF-IDF por autor, agregando los textos de todos sus artículos.

La base original no se modifica. La carpeta de salida esperada es:

```text
C:\Users\hazar\Desktop\PROYECTO\06_LLM\01_Datos
```


## 1. Importar librerías

In [1]:
from pathlib import Path
import sqlite3
import re

import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer


## 2. Configuración de rutas

In [ ]:

RAIZ_PROYECTO = Path(r"C:\Users\hazar\Desktop\PROYECTO")


RUTA_CSV = RAIZ_PROYECTO / "05_Correcciones" / "UNAM_Completo_2024_2025.csv"


CARPETA_SALIDA = RAIZ_PROYECTO / "06_LLM" / "01_Datos"


if not RUTA_CSV.exists():
    ruta_prueba = Path("/mnt/data/UNAM_Completo_2024_2025.csv")
    if ruta_prueba.exists():
        RUTA_CSV = ruta_prueba
        CARPETA_SALIDA = Path("/mnt/data/06_LLM_01_Datos")

CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

RUTA_SQLITE = CARPETA_SALIDA / "UNAM_Completo_2024_2025.sqlite"
RUTA_TFIDF_ARTICULOS = CARPETA_SALIDA / "tfidf_articulos.joblib"
RUTA_TFIDF_AUTORES = CARPETA_SALIDA / "tfidf_autores.joblib"

print("CSV de entrada:", RUTA_CSV)
print("Carpeta de salida:", CARPETA_SALIDA)
print("Archivos que se crearán:")
for ruta in [RUTA_SQLITE, RUTA_TFIDF_ARTICULOS, RUTA_TFIDF_AUTORES]:
    print(" -", ruta.name)


CSV de entrada: C:\Users\hazar\Desktop\PROYECTO\05_Correcciones\UNAM_Completo_2024_2025.csv
Carpeta de salida: C:\Users\hazar\Desktop\PROYECTO\06_LLM\01_Datos
Archivos que se crearán:
 - UNAM_Completo_2024_2025.sqlite
 - tfidf_articulos.joblib
 - tfidf_autores.joblib


## 3. Cargar y validar la base canónica

Se valida que existan exactamente las 14 columnas canónicas del proyecto.

In [3]:
COLUMNAS_CANONICAS = [
    "indice",
    "Titulo",
    "Año",
    "Autor_norm",
    "Afiliacion1",
    "Afiliacion2",
    "ISBN",
    "ISSN",
    "Doi",
    "URL",
    "Area",
    "Subarea",
    "Keywords",
    "Abstract",
]

if not RUTA_CSV.exists():
    raise FileNotFoundError(f"No se encontró el archivo de entrada: {RUTA_CSV}")

df = pd.read_csv(RUTA_CSV, dtype=str, keep_default_na=False, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]

faltantes = [c for c in COLUMNAS_CANONICAS if c not in df.columns]
extras = [c for c in df.columns if c not in COLUMNAS_CANONICAS]

if faltantes:
    raise ValueError(f"Faltan columnas canónicas: {faltantes}")
if extras:
    raise ValueError(f"Hay columnas extra no canónicas: {extras}")

df = df[COLUMNAS_CANONICAS].copy()

# Limpieza mínima de espacios para evitar problemas de consulta.
# No se normalizan contenidos ni se alteran los criterios de curaduría.
for col in df.columns:
    df[col] = df[col].astype(str).str.strip()

# Para SQLite conviene que indice y Año se puedan consultar como enteros.
df["indice"] = pd.to_numeric(df["indice"], errors="coerce").astype("Int64")
df["Año"] = pd.to_numeric(df["Año"], errors="coerce").astype("Int64")

if df["indice"].isna().any():
    raise ValueError("Hay valores no numéricos o vacíos en la columna indice.")
if df["Año"].isna().any():
    raise ValueError("Hay valores no numéricos o vacíos en la columna Año.")

print("Dimensiones:", df.shape)
print("Artículos únicos:", df["indice"].nunique())
print("Autores únicos:", df["Autor_norm"].nunique())
df.head()


Dimensiones: (905, 14)
Artículos únicos: 406
Autores únicos: 550


,indice,Titulo,Año,Autor_norm,Afiliacion1,Afiliacion2,ISBN,ISSN,Doi,URL,Area,Subarea,Keywords,Abstract
0,1,A Big Data: Oriented Spatial Data Infrastructu...,2024,Marco Antonio Lopez Vega,Instituto de Geografía,,978-3-031-53959-6,2367-3370,10.1007/978-3-031-53960-2_36,https://www.scopus.com/pages/publications/8518...,ISBD,,"big data, lanot, mongodb, sdi, spatial data sc...",The overall framework under which a Geographic...
1,1,A Big Data: Oriented Spatial Data Infrastructu...,2024,Ranulfo Rodriguez Sobreyra,Instituto de Ciencias del Mar y Limnología,,978-3-031-53959-6,2367-3370,10.1007/978-3-031-53960-2_36,https://www.scopus.com/pages/publications/8518...,ISBD,,"big data, lanot, mongodb, sdi, spatial data sc...",The overall framework under which a Geographic...
2,1,A Big Data: Oriented Spatial Data Infrastructu...,2024,Stephane Couturier,Instituto de Geografía,,978-3-031-53959-6,2367-3370,10.1007/978-3-031-53960-2_36,https://www.scopus.com/pages/publications/8518...,ISBD,,"big data, lanot, mongodb, sdi, spatial data sc...",The overall framework under which a Geographic...
3,2,A Culturally-Aware AI Tool for Crowdworkers: L...,2024,Saiph Savage,Universidad Nacional Autonoma de Mexico,,,2573-0142,10.1145/3686899,https://doi.org/10.1145/3686899,ISBD,,"ai, chronemics, crowdsourcing, culture, system","Crowdsourcing markets are expanding worldwide,..."
4,3,A Systematic Literature Review of 10 years of ...,2024,Everardo Barcenas,Facultad de Ingeniería,,,0361-7688,10.1134/s0361768824700737,https://www.scopus.com/pages/publications/8521...,ISBD,,"language models, software requirements specifi...",Program synthesis is the process of automatica...


## 4. Crear SQLite con una sola tabla canónica

La base SQLite conserva una sola tabla principal: `articulos_autores`.

Esto mantiene la lógica original de la base curada: una fila representa una relación **artículo–autor–afiliación**.

In [4]:
# Eliminar únicamente salidas previas esperadas, para no generar duplicados.
for ruta in [RUTA_SQLITE, RUTA_TFIDF_ARTICULOS, RUTA_TFIDF_AUTORES]:
    if ruta.exists():
        ruta.unlink()

# Crear base SQLite.
with sqlite3.connect(RUTA_SQLITE) as conn:
    df.to_sql("articulos_autores", conn, if_exists="replace", index=False)

    # Índices internos para acelerar consultas frecuentes.
    conn.execute("CREATE INDEX IF NOT EXISTS idx_articulos_indice ON articulos_autores(indice);")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_articulos_autor ON articulos_autores(Autor_norm);")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_articulos_area ON articulos_autores(Area);")
    conn.execute('CREATE INDEX IF NOT EXISTS idx_articulos_anio ON articulos_autores("Año");')
    conn.execute("CREATE INDEX IF NOT EXISTS idx_articulos_titulo ON articulos_autores(Titulo);")

    total_sql = conn.execute("SELECT COUNT(*) FROM articulos_autores;").fetchone()[0]
    total_articulos_sql = conn.execute("SELECT COUNT(DISTINCT indice) FROM articulos_autores;").fetchone()[0]
    total_autores_sql = conn.execute("SELECT COUNT(DISTINCT Autor_norm) FROM articulos_autores;").fetchone()[0]

print("SQLite creado:", RUTA_SQLITE)
print("Filas en SQLite:", total_sql)
print("Artículos en SQLite:", total_articulos_sql)
print("Autores en SQLite:", total_autores_sql)


SQLite creado: C:\Users\hazar\Desktop\PROYECTO\06_LLM\01_Datos\UNAM_Completo_2024_2025.sqlite
Filas en SQLite: 905
Artículos en SQLite: 406
Autores en SQLite: 550


## 5. Preparar textos para TF-IDF

Para obtener mejores resultados en preguntas temáticas, se usa una representación combinada y ponderada:

```text
Titulo   x3
Keywords x4
Abstract x1
```

La razón es que las `Keywords` suelen ser el campo más directo para similitud temática, el `Titulo` resume el tema del artículo y el `Abstract` aporta contexto amplio.

In [5]:
def texto_valido(valor) -> str:
    """Convierte valores vacíos a cadena limpia."""
    if pd.isna(valor):
        return ""
    texto = str(valor).strip()
    if texto.lower() in {"nan", "none", "null", "na", "n/a"}:
        return ""
    return texto


def unir_unicos(valores, separador="; ") -> str:
    """Une valores únicos no vacíos conservando el primer orden de aparición."""
    vistos = set()
    salida = []
    for valor in valores:
        texto = texto_valido(valor)
        if not texto:
            continue
        clave = texto.lower()
        if clave not in vistos:
            vistos.add(clave)
            salida.append(texto)
    return separador.join(salida)


def texto_articulo(titulo, keywords, abstract) -> str:
    """Texto ponderado para representar un artículo en TF-IDF."""
    titulo = texto_valido(titulo)
    keywords = texto_valido(keywords)
    abstract = texto_valido(abstract)
    partes = []
    if titulo:
        partes.extend([titulo] * 3)
    if keywords:
        partes.extend([keywords] * 4)
    if abstract:
        partes.append(abstract)
    return " ".join(partes)


## 6. Crear índice TF-IDF por artículo

Cada artículo aparece una sola vez y se identifica por `indice`.

In [6]:
# Una fila por artículo.
# Se usa el primer valor no vacío para metadatos repetidos por autor.
orden_articulos = df.sort_values(["indice", "Autor_norm"]).copy()

articulos = (
    orden_articulos
    .groupby("indice", as_index=False)
    .agg({
        "Titulo": "first",
        "Año": "first",
        "Area": "first",
        "Doi": "first",
        "URL": "first",
        "ISBN": "first",
        "ISSN": "first",
        "Keywords": "first",
        "Abstract": "first",
        "Autor_norm": lambda x: unir_unicos(x, "; "),
        "Afiliacion1": lambda x: unir_unicos(x, "; "),
        "Afiliacion2": lambda x: unir_unicos(x, "; "),
    })
    .rename(columns={
        "Autor_norm": "Autores",
        "Afiliacion1": "Afiliaciones1",
        "Afiliacion2": "Afiliaciones2",
    })
)

articulos["texto_tfidf"] = articulos.apply(
    lambda r: texto_articulo(r["Titulo"], r["Keywords"], r["Abstract"]), axis=1
)

print("Artículos para TF-IDF:", articulos.shape[0])
articulos[["indice", "Titulo", "Area", "texto_tfidf"]].head(3)


Artículos para TF-IDF: 406


,indice,Titulo,Area,texto_tfidf
0,1,A Big Data: Oriented Spatial Data Infrastructu...,ISBD,A Big Data: Oriented Spatial Data Infrastructu...
1,2,A Culturally-Aware AI Tool for Crowdworkers: L...,ISBD,A Culturally-Aware AI Tool for Crowdworkers: L...
2,3,A Systematic Literature Review of 10 years of ...,ISBD,A Systematic Literature Review of 10 years of ...


## 7. Crear índice TF-IDF por autor

Para cada autor se agregan los textos de todos sus artículos. Esto permite responder preguntas de similitud temática entre autores, incluso si no han colaborado.

In [7]:
df_autor_texto = df.copy()
df_autor_texto["texto_fila"] = df_autor_texto.apply(
    lambda r: texto_articulo(r["Titulo"], r["Keywords"], r["Abstract"]), axis=1
)

autores = (
    df_autor_texto
    .groupby("Autor_norm", as_index=False)
    .agg({
        "indice": lambda x: unir_unicos([str(v) for v in x], "; "),
        "Titulo": lambda x: unir_unicos(x, "; "),
        "Area": lambda x: unir_unicos(x, "; "),
        "Afiliacion1": lambda x: unir_unicos(x, "; "),
        "Afiliacion2": lambda x: unir_unicos(x, "; "),
        "Keywords": lambda x: unir_unicos(x, "; "),
        "texto_fila": lambda x: " ".join([texto_valido(v) for v in x if texto_valido(v)]),
    })
    .rename(columns={
        "indice": "indices",
        "Titulo": "Titulos",
        "Area": "Areas",
        "Afiliacion1": "Afiliaciones1",
        "Afiliacion2": "Afiliaciones2",
        "Keywords": "Keywords_agregadas",
        "texto_fila": "texto_tfidf",
    })
)

autores["num_articulos"] = autores["indices"].apply(lambda x: len([p for p in str(x).split("; ") if p]))

autores = autores[[
    "Autor_norm", "num_articulos", "indices", "Titulos", "Areas",
    "Afiliaciones1", "Afiliaciones2", "Keywords_agregadas", "texto_tfidf"
]]

print("Autores para TF-IDF:", autores.shape[0])
autores.head(3)


Autores para TF-IDF: 550


,Autor_norm,num_articulos,indices,Titulos,Areas,Afiliaciones1,Afiliaciones2,Keywords_agregadas,texto_tfidf
0,A Andres,1,314,Improving Gamma-ray Source Searches with Image...,SIAV,Instituto de Física,,"blind source separation, cosmology, germanium ...",Improving Gamma-ray Source Searches with Image...
1,A Bernal,1,314,Improving Gamma-ray Source Searches with Image...,SIAV,Instituto de Astronomía,,"blind source separation, cosmology, germanium ...",Improving Gamma-ray Source Searches with Image...
2,A L Padilla Ortiz,1,295,Edge Computing System for Automatic Detection ...,SIAV,Instituto de Ciencias Aplicadas y Tecnología,,"cdr, copd, edge computing, machine learning",Edge Computing System for Automatic Detection ...


## 8. Vectorizar con TF-IDF

Se ajusta un solo vectorizador sobre artículos y autores para que ambos queden en el mismo espacio vectorial.

In [8]:
STOPWORDS_ES_EN = {
    # Español básico
    "a", "al", "algo", "ante", "antes", "como", "con", "contra", "cual", "cuando", "de", "del",
    "desde", "donde", "dos", "el", "ella", "ellas", "ellos", "en", "entre", "era", "eran", "es",
    "esa", "esas", "ese", "eso", "esos", "esta", "estaba", "estado", "estan", "estar", "este", "esto",
    "estos", "fue", "ha", "han", "hasta", "hay", "la", "las", "lo", "los", "mas", "me", "mi", "muy",
    "no", "nos", "o", "para", "pero", "por", "que", "se", "sin", "sobre", "son", "su", "sus", "tambien",
    "un", "una", "uno", "unos", "y", "ya",
    # Inglés básico
    "an", "and", "are", "as", "at", "be", "been", "by", "for", "from", "has", "have", "in",
    "into", "is", "it", "its", "of", "on", "or", "that", "the", "their", "these", "this", "to", "using",
    "was", "we", "were", "with", "within", "without", "which", "our", "can", "may", "also",
}

corpus_articulos = articulos["texto_tfidf"].fillna("").tolist()
corpus_autores = autores["texto_tfidf"].fillna("").tolist()
corpus_total = corpus_articulos + corpus_autores

vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    stop_words=list(STOPWORDS_ES_EN),
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
    max_features=30000,
    sublinear_tf=True,
    norm="l2",
)

vectorizer.fit(corpus_total)

matriz_articulos = vectorizer.transform(corpus_articulos)
matriz_autores = vectorizer.transform(corpus_autores)

print("Matriz artículos:", matriz_articulos.shape)
print("Matriz autores:", matriz_autores.shape)
print("Términos en vocabulario:", len(vectorizer.vocabulary_))


Matriz artículos: (406, 30000)
Matriz autores: (550, 30000)
Términos en vocabulario: 30000


## 9. Guardar exactamente tres archivos

La carpeta de salida recibirá sólo los tres recursos necesarios para la siguiente libreta:

```text
UNAM_Completo_2024_2025.sqlite
tfidf_articulos.joblib
tfidf_autores.joblib
```


In [9]:
# Metadatos compactos pero suficientes para recuperar resultados en la siguiente libreta.
metadata_articulos = articulos[[
    "indice", "Titulo", "Año", "Area", "Doi", "URL", "ISBN", "ISSN",
    "Autores", "Afiliaciones1", "Afiliaciones2", "Keywords", "Abstract"
]].copy()

metadata_autores = autores[[
    "Autor_norm", "num_articulos", "indices", "Titulos", "Areas",
    "Afiliaciones1", "Afiliaciones2", "Keywords_agregadas"
]].copy()

objeto_tfidf_articulos = {
    "tipo": "tfidf_articulos",
    "descripcion": "Índice TF-IDF por artículo construido con Titulo x3 + Keywords x4 + Abstract x1.",
    "campos_texto": ["Titulo", "Keywords", "Abstract"],
    "ponderacion": {"Titulo": 3, "Keywords": 4, "Abstract": 1},
    "vectorizer": vectorizer,
    "matrix": matriz_articulos,
    "metadata": metadata_articulos,
}

objeto_tfidf_autores = {
    "tipo": "tfidf_autores",
    "descripcion": "Índice TF-IDF por autor construido agregando los textos de todos sus artículos.",
    "campos_texto": ["Titulo", "Keywords", "Abstract"],
    "ponderacion": {"Titulo": 3, "Keywords": 4, "Abstract": 1},
    "vectorizer": vectorizer,
    "matrix": matriz_autores,
    "metadata": metadata_autores,
}

joblib.dump(objeto_tfidf_articulos, RUTA_TFIDF_ARTICULOS, compress=3)
joblib.dump(objeto_tfidf_autores, RUTA_TFIDF_AUTORES, compress=3)

archivos_generados = [RUTA_SQLITE, RUTA_TFIDF_ARTICULOS, RUTA_TFIDF_AUTORES]

print("Archivos generados:")
for ruta in archivos_generados:
    print(f"- {ruta.name} | {ruta.stat().st_size / (1024*1024):.2f} MB")


Archivos generados:
- UNAM_Completo_2024_2025.sqlite | 2.11 MB
- tfidf_articulos.joblib | 1.01 MB
- tfidf_autores.joblib | 1.25 MB


## 10. Validación rápida

La siguiente validación sólo imprime resultados dentro de la libreta. No genera archivos adicionales.

In [10]:
with sqlite3.connect(RUTA_SQLITE) as conn:
    tablas = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
    columnas_sql = pd.read_sql_query("PRAGMA table_info(articulos_autores);", conn)
    filas_sql = conn.execute("SELECT COUNT(*) FROM articulos_autores;").fetchone()[0]
    articulos_sql = conn.execute("SELECT COUNT(DISTINCT indice) FROM articulos_autores;").fetchone()[0]
    autores_sql = conn.execute("SELECT COUNT(DISTINCT Autor_norm) FROM articulos_autores;").fetchone()[0]

print("Tablas en SQLite:", tablas["name"].tolist())
print("Columnas en articulos_autores:", columnas_sql["name"].tolist())
print("Filas SQLite:", filas_sql)
print("Artículos SQLite:", articulos_sql)
print("Autores SQLite:", autores_sql)
print("Matriz TF-IDF artículos:", matriz_articulos.shape)
print("Matriz TF-IDF autores:", matriz_autores.shape)
print("Salida lista en:", CARPETA_SALIDA)


Tablas en SQLite: ['articulos_autores']
Columnas en articulos_autores: ['indice', 'Titulo', 'Año', 'Autor_norm', 'Afiliacion1', 'Afiliacion2', 'ISBN', 'ISSN', 'Doi', 'URL', 'Area', 'Subarea', 'Keywords', 'Abstract']
Filas SQLite: 905
Artículos SQLite: 406
Autores SQLite: 550
Matriz TF-IDF artículos: (406, 30000)
Matriz TF-IDF autores: (550, 30000)
Salida lista en: C:\Users\hazar\Desktop\PROYECTO\06_LLM\01_Datos
